# Example 3.4.1 / 1: Write a Report File

Companion notebook for **3.4.1 Output Files and Logging** in the AMPL lab Documentation, section "Write a Report File". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Write a Report File

The run below solves the production model, writes a header with `>`, appends one row per
product with `>>`, closes the file, and then reads it back from Python:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # Products
param profit {PROD};                      # Profit per ton
param market {PROD} >= 0;                 # Maximum sales in tons
var Make {p in PROD} >= 0, <= market[p];  # Tons to make, capped by sales
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # Total profit in dollars

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
ampl.eval('printf "product,amount\\n" > "plan.csv";')
ampl.eval('printf {p in PROD} "%s,%.6g\\n", p, Make[p] >> "plan.csv";')
ampl.eval('close "plan.csv";')
with open("plan.csv") as report:
    print(report.read(), end="")

product,amount
bands,6000
coils,4000
